In [ ]:
# !pip install --upgrade pip setuptools wheel

In [ ]:
# !pip install --upgrade build

In [ ]:
# !apt-get install -y -qq libboost-all-dev cmake build-essential

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
#코랩 환경에서는 openCL GPU
# 필수 시스템 패키지 설치
# !apt-get install -y -qq libboost-all-dev cmake build-essential

# # 기존 설치 제거
# !pip uninstall -y lightgbm
# !rm -rf /content/LightGBM

# # ✅ LightGBM 3.2.1 (또는 3.2.0) 클론 - setup.py 기반
# !git clone --branch v3.2.1 --recursive https://github.com/microsoft/LightGBM.git /content/LightGBM

# # GPU 빌드
# %cd /content/LightGBM
# !mkdir build
# %cd build
# !cmake -DUSE_GPU=1 ..
# !make -j4

# # 파이썬 패키지 설치 (setup.py 존재함)
# %cd ../python-package
# !ls  # ← 여기서 setup.py 파일이 보여야 성공 lightgbm  MANIFEST.in  README.rst  setup.py
# !pip install .

Cloning into '/content/LightGBM'...
remote: Enumerating objects: 35433, done.
remote: Counting objects: 100% (201/201), done.
remote: Compressing objects: 100% (124/124), done.
remote: Total 35433 (delta 134), reused 77 (delta 77), pack-reused 35232 (from 4)
Receiving objects: 100% (35433/35433), 24.02 MiB | 29.04 MiB/s, done.
Resolving deltas: 100% (26333/26333), done.
Note: switching to 'b8e38ec1eb8020052d5b39e31e9f2cb6366fb873'.

You are in 'detached HEAD' state. You can look around, make experimental
changes and commit them, and you can discard any commits you make in this
state without impacting any branches by switching back to a branch.

If you want to create a new branch to retain commits you create, you may
do so (now or later) by using -c with the switch command. Example:

  git switch -c <new-branch-name>

Or undo this operation with:

  git switch -

Turn off this advice by setting config variable advice.detachedHead to false

Submodule 'include/boost/compute' (https://gith

In [1]:
!pip uninstall -y lightgbm
!rm -rf /content/LightGBM
!pip install lightgbm

Found existing installation: lightgbm 4.5.0
Uninstalling lightgbm-4.5.0:
  Successfully uninstalled lightgbm-4.5.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 70.1 MB/s eta 0:00:00


In [2]:
# 설치 확인
import lightgbm as lgb
print(lgb.__version__)

4.6.0


In [3]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 395.9/395.9 kB 17.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.7/242.7 kB 18.4 MB/s eta 0:00:00


In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import optuna
import os
import lightgbm as lgb

import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from sklearn.metrics import accuracy_score

# 경고 뜨지 않게 설정
import warnings
warnings.filterwarnings('ignore')

# 그래프 설정
sns.set()

# 그래프 기본 설정
plt.rcParams['font.family'] = 'NanumGothic'
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['figure.figsize'] = 12, 6
plt.rcParams['font.size'] = 14
plt.rcParams['axes.unicode_minus'] = False

In [6]:
# 데이터 경로
print(os.getcwd())
path = '/content/drive/MyDrive/파이널 프로젝트/2025_07_07/'
print(os.listdir(path))

/content
['Segment_merge_ver_05.parquet', 'Segment_merge_test_ver_05.parquet']


In [7]:
train = pd.read_parquet(path + 'Segment_merge_ver_05.parquet')

In [8]:
test = pd.read_parquet(path +'Segment_merge_test_ver_05.parquet' ) # 기준년월, ID 컬럼 삭제된 파일

In [9]:
print(train.shape)
print(test.shape)

(2400000, 138)
(600000, 137)


In [10]:
drop_cols = ['기준년월','ID','Segment']
x = train.drop(columns=drop_cols)
y = train['Segment']

In [11]:
# 데이터 분할
X_tr, X_val, y_tr, y_val = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

# x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

In [12]:
# # 파라미터 값 조정
# def objective(trial):
#     param = {
#         'objective': 'multiclass',
#         'num_class': 5,
#         'metric': 'multi_logloss',
#         'verbosity': -1,
#         'boosting_type': 'gbdt',
#         'learning_rate': trial.suggest_float('learning_rate', 0.05, 0.12),  # 좁힘
#         'num_leaves': trial.suggest_int('num_leaves', 150, 250),           # 좁힘
#         'max_depth': trial.suggest_int('max_depth', 10, 15),               # 최적값에 집중
#         'min_child_samples': trial.suggest_int('min_child_samples', 20, 50),
#         'subsample': trial.suggest_float('subsample', 0.8, 0.95),
#         'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 0.9),
#         'reg_alpha': trial.suggest_float('reg_alpha', 1e-4, 1.0, log=True), # 과도한 제약 제거
#         'reg_lambda': trial.suggest_float('reg_lambda', 1e-5, 0.1, log=True),
#         'n_estimators': 1000,
#         'random_state': 42,
#         'n_jobs': -1,
#          'device_type':'gpu'
#     }

#     model = lgb.LGBMClassifier(**param)

#     model.fit(
#         X_tr, y_tr,
#         eval_set=[(X_val, y_val)],
#         eval_metric='multi_logloss',
#         callbacks=[lgb.early_stopping(50, verbose=False)],
#     )

#     preds = model.predict(X_val)
#     acc = accuracy_score(y_val, preds)
#     return acc

#     [W 2025-07-07 06:35:57,773] Trial 10 failed with parameters: {'learning_rate': 0.08290312337050593, 'num_leaves': 207, 'max_depth': 11, 'min_child_samples': 48, 'subsample': 0.9441386022205047, 'colsample_bytree': 0.8781227616631291, 'reg_alpha': 0.4168762731451913, 'reg_lambda': 2.999104616193986e-05} because of the following error: LightGBMError('Check failed: (best_split_info.left_count) > (0) at /tmp/lightgbm/LightGBM/lightgbm-python/src/treelearner/serial_tree_learner.cpp, line 846 .\n').
# Traceback (most recent call last):
#   File "/usr/local/lib/python3.11/dist-packages/optuna/study/_optimize.py", line 201, in _run_trial
#     value_or_values = func(trial)
#                       ^^^^^^^^^^^
#   File "/tmp/ipython-input-11-1016100464.py", line 25, in objective
#     model.fit(
#   File "/usr/local/lib/python3.11/dist-packages/lightgbm/sklearn.py", line 1284, in fit
#     super().fit(
#   File "/usr/local/lib/python3.11/dist-packages/lightgbm/sklearn.py", line 955, in fit
#     self._Booster = train(
#                     ^^^^^^
#   File "/usr/local/lib/python3.11/dist-packages/lightgbm/engine.py", line 307, in train
#     booster.update(fobj=fobj)
#   File "/usr/local/lib/python3.11/dist-packages/lightgbm/basic.py", line 4135, in update
#     _safe_call(
#   File "/usr/local/lib/python3.11/dist-packages/lightgbm/basic.py", line 296, in _safe_call
#     raise LightGBMError(_LIB.LGBM_GetLastError().decode("utf-8"))
# lightgbm.basic.LightGBMError: Check failed: (best_split_info.left_count) > (0) at /tmp/lightgbm/LightGBM/lightgbm-python/src/treelearner/serial_tree_learner.cpp, line 846 .


In [16]:
# objective 함수 정의
def objective(trial):
  param = {
    'objective': 'multiclass',
    'num_class': 5,
    'metric': 'multi_logloss',
    'verbosity': -1,
    'boosting_type': 'gbdt',
    'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
    'num_leaves': trial.suggest_int('num_leaves', 31, 128),            # 최대값 축소 (과도한 복잡도 방지)
    'max_depth': trial.suggest_int('max_depth', 7, 20),                # 최소값 조금 올림, 최대값 축소
    'min_child_samples': trial.suggest_int('min_child_samples', 20, 50),# 최소값 줄여서 분할 가능성 증가
    'subsample': trial.suggest_float('subsample', 0.7, 1.0),           # 기본값 유지하거나 조금 좁히기
    'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
    'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 1.0, log=True),  # 너무 큰 값 제외
    'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 1.0, log=True),
    'n_estimators': 1000,
    'random_state': 42,
    'n_jobs': -1
}
  model = lgb.LGBMClassifier(**param,verbose=1)

  model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        eval_metric='multi_logloss',
        callbacks=[lgb.early_stopping(50, verbose=False)],
    )

  preds = model.predict(X_val)
  acc = accuracy_score(y_val, preds)
  return acc  # maximize accuracy

In [19]:
study = optuna.create_study(direction='maximize')

# # 예시: 병렬 실행 (최대 4개 프로세스)
# optuna.study.create_study(direction='maximize')
# study.optimize(objective, n_trials=50, n_jobs=4)

[I 2025-07-08 00:18:49,728] A new study created in memory with name: no-name-32c5f4ac-5d19-428a-bf23-b825978c1044


ver.03파일은 에러가나서 분석 중단
병렬 처리를 해서 에러가나나...? <br>
params 값 범위 조정 X + n_jobs 옵션 X 실행해봄 <br>
에러가 발생하는 주 원인 중 하나는 min_child_samples가 너무 크거나, num_leaves와 max_depth 조합이 적절하지 않아서 분할 시 왼쪽 자식 노드에 데이터가 없을 때입니다.

In [20]:
study.optimize(objective, n_trials=30)

print("Best trial:")
print(" Value: ", study.best_value)
print(" Params: ")

[I 2025-07-08 00:20:29,324] Trial 0 finished with value: 0.88664375 and parameters: {'learning_rate': 0.07466701696176672, 'num_leaves': 36, 'max_depth': 15, 'min_child_samples': 42, 'subsample': 0.7322354108006245, 'colsample_bytree': 0.992485321261734, 'reg_alpha': 0.02382648722197509, 'reg_lambda': 2.9005835528991715e-05}. Best is trial 0 with value: 0.88664375.
[I 2025-07-08 00:36:14,141] Trial 1 finished with value: 0.93765 and parameters: {'learning_rate': 0.07908925401862119, 'num_leaves': 119, 'max_depth': 9, 'min_child_samples': 41, 'subsample': 0.8365979872487723, 'colsample_bytree': 0.9170262821764648, 'reg_alpha': 0.008998855108257963, 'reg_lambda': 7.003764272106548e-06}. Best is trial 1 with value: 0.93765.
[I 2025-07-08 00:37:54,806] Trial 2 finished with value: 0.89005 and parameters: {'learning_rate': 0.08635816013953589, 'num_leaves': 54, 'max_depth': 14, 'min_child_samples': 46, 'subsample': 0.9313315725594427, 'colsample_bytree': 0.8238908678585743, 'reg_alpha': 0.0

Best trial:
 Value:  0.9471354166666667
 Params: 


In [ ]:
# print(model.get_params()['device_type']) #gpu 사용여부

In [21]:
# 최적 파라미터로 전체 학습
best_params = study.best_params

final_model = lgb.LGBMClassifier(**best_params)
# X_tr, X_val, y_tr, y_val
final_model.fit( X_tr, y_tr)

LGBMClassifier(colsample_bytree=0.7943406603089208,
               learning_rate=0.09968074420540968, max_depth=12,
               min_child_samples=44, num_leaves=127,
               reg_alpha=0.008330746123943621, reg_lambda=0.6228954923795628,
               subsample=0.7803826377746543)

In [29]:
import pickle
pickle.dump(final_model, open('/content/drive/MyDrive/Colab_Notebooks/LGBM_model.pkl', 'wb'))

In [30]:
print(os.getcwd())

/content


In [31]:
X_test = test.drop(columns=['기준년월','ID'])

In [32]:
## 예측
# 5. 예측 수행
final_pred = final_model.predict(X_test)               # 예측 레이블
final_proba = final_model.predict_proba(X_test)

In [33]:
# 6. 결과 저장
res_df = pd.DataFrame({
    'ID': test.ID,
    'Segment': final_pred
})

In [35]:
res_df.to_csv('/content/drive/MyDrive/Colab_Notebooks/hyper_pred_ver_05_result.csv')

In [34]:
representative_segments = (
    res_df.groupby('ID')['Segment']
    .agg(lambda x: x.mode().iloc[0])  # mode()는 여러 개일 수 있어 .iloc[0]으로 하나 선택
    .reset_index()
    .rename(columns={'Segment': 'Segment'})
)

print(representative_segments)

               ID Segment
0      TEST_00000       E
1      TEST_00001       E
2      TEST_00002       E
3      TEST_00003       E
4      TEST_00004       E
...           ...     ...
99995  TEST_99995       E
99996  TEST_99996       E
99997  TEST_99997       E
99998  TEST_99998       C
99999  TEST_99999       E

[100000 rows x 2 columns]


In [36]:
representative_segments.shape

(100000, 2)

In [37]:
representative_segments.to_csv('/content/drive/MyDrive/Colab_Notebooks/hyper_pred_ver_05_Segment.csv',index=False)

전처리 ver.02 파일 하이퍼 파라미터 코드 및 결과

In [ ]:
# # objective 함수 정의
# def objective(trial):
#     param = {
#         'objective': 'multiclass',
#         'num_class': 5,
#         'metric': 'multi_logloss',
#         'verbosity': -1,
#         'boosting_type': 'gbdt',
#         'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
#         'num_leaves': trial.suggest_int('num_leaves', 31, 256),
#         'max_depth': trial.suggest_int('max_depth', 3, 15),
#         'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
#         'subsample': trial.suggest_float('subsample', 0.6, 1.0),
#         'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
#         'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
#         'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
#         'n_estimators': 1000,
#         'random_state': 42,
#         'n_jobs': -1
#     }

#     model = lgb.LGBMClassifier(**param)

#     model.fit(
#         X_tr, y_tr,
#         eval_set=[(X_val, y_val)],
#         eval_metric='multi_logloss',
#         callbacks=[lgb.early_stopping(50, verbose=False)],
#     )

#     preds = model.predict(X_val)
#     acc = accuracy_score(y_val, preds)
#     return acc  # maximize accuracy

In [ ]:
study = optuna.create_study(direction='maximize')

[I 2025-07-04 02:20:24,179] A new study created in memory with name: no-name-3ca0ddd9-adfb-49e4-8f40-5ae5c47a3614


In [ ]:
study.optimize(objective, n_trials=30)

print("Best trial:")
print(" Value: ", study.best_value)
print(" Params: ")


[I 2025-07-04 02:30:29,848] Trial 0 finished with value: 0.883425 and parameters: {'learning_rate': 0.034638243239393386, 'num_leaves': 170, 'max_depth': 3, 'min_child_samples': 56, 'subsample': 0.9998793510407011, 'colsample_bytree': 0.9598155319199569, 'reg_alpha': 2.027879221439409e-07, 'reg_lambda': 4.236926716050374e-06}. Best is trial 0 with value: 0.883425.
[I 2025-07-04 02:35:21,008] Trial 1 finished with value: 0.89506875 and parameters: {'learning_rate': 0.05494596683922364, 'num_leaves': 202, 'max_depth': 13, 'min_child_samples': 81, 'subsample': 0.6781913086860935, 'colsample_bytree': 0.7930060051856186, 'reg_alpha': 8.763968689532738e-05, 'reg_lambda': 8.304535294271677e-08}. Best is trial 1 with value: 0.89506875.
[I 2025-07-04 02:43:02,976] Trial 2 finished with value: 0.8830145833333334 and parameters: {'learning_rate': 0.030554750786629677, 'num_leaves': 176, 'max_depth': 3, 'min_child_samples': 27, 'subsample': 0.6177858184581875, 'colsample_bytree': 0.806318469727666

Best trial:
 Value:  0.9391895833333334
 Params: 
    learning_rate: 0.09134875615222389
    num_leaves: 199
    max_depth: 13
    min_child_samples: 34
    subsample: 0.8769232988448797
    colsample_bytree: 0.7425736208006023
    reg_alpha: 0.25572452150806746
    reg_lambda: 0.0006117169730699716


In [ ]:
# 최적 파라미터로 전체 학습
best_params = study.best_params

final_model = lgb.LGBMClassifier(**best_params)
# X_tr, X_val, y_tr, y_val
final_model.fit( X_tr, y_tr)

LGBMClassifier(colsample_bytree=0.7425736208006023,
               learning_rate=0.09134875615222389, max_depth=13,
               min_child_samples=34, num_leaves=199,
               reg_alpha=0.25572452150806746, reg_lambda=0.0006117169730699716,
               subsample=0.8769232988448797)

In [ ]:
X_test = test.drop(columns=['기준년월','ID'])

In [ ]:
## 예측
# 5. 예측 수행
final_pred = final_model.predict(X_test)               # 예측 레이블
final_proba = final_model.predict_proba(X_test)

In [ ]:
# 6. 결과 저장
res_df = pd.DataFrame({
    'ID': test.ID,
    'Segment': final_pred
})

In [ ]:
res_df.to_csv('/content/drive/MyDrive/Colab_Notebooks/01.like_lion_final_prj/hyper_pred_result.csv')

In [ ]:
res_df.groupby('ID')['Segment']
    .agg(lambda x: x.mode())

IndentationError: unexpected indent (ipython-input-45-2163916041.py, line 2)

In [ ]:
representative_segments = (
    res_df.groupby('ID')['Segment']
    .agg(lambda x: x.mode().iloc[0])  # mode()는 여러 개일 수 있어 .iloc[0]으로 하나 선택
    .reset_index()
    .rename(columns={'Segment': 'Segment'})
)

print(representative_segments)

               ID Segment
0      TEST_00000       E
1      TEST_00001       D
2      TEST_00002       D
3      TEST_00003       E
4      TEST_00004       E
...           ...     ...
99995  TEST_99995       E
99996  TEST_99996       E
99997  TEST_99997       E
99998  TEST_99998       C
99999  TEST_99999       E

[100000 rows x 2 columns]


In [ ]:
representative_segments.to_csv('/content/drive/MyDrive/Colab_Notebooks/01.like_lion_final_prj/submission_1.csv', index=False)



*   항목 추가
*   항목 추가



In [ ]:
res_df.Segment.value_counts()

,count
Segment,
E,495356
D,77128
C,25974
B,806
A,736


In [ ]:
res_df[res_df.Segment == 'A']

,ID,Segment
631,TEST_00631,A
1194,TEST_01194,A
2166,TEST_02166,A
2254,TEST_02254,A
3387,TEST_03387,A
...,...,...
596571,TEST_96571,A
596614,TEST_96614,A
597520,TEST_97520,A
599537,TEST_99537,A
